# Ariadne Programmatic API

Complete walkthrough of the Scala API for creating, querying, joining, and managing indexes.

In [1]:
%%init_spark
launcher.master = "local[*]"
launcher.conf.set("spark.ariadne.storagePath", "/home/ariadne_storage")
launcher.conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

In [2]:
import dev.cjfravel.ariadne.{Index, IndexCatalog}
import dev.cjfravel.ariadne.Index.DataFrameOps
import org.apache.spark.sql.types._
import org.apache.spark.sql.Row
implicit val spark = org.apache.spark.sql.SparkSession.builder().getOrCreate()

// Clean up any previous index data
import org.apache.hadoop.fs.{FileSystem, Path}
val fs = FileSystem.get(spark.sparkContext.hadoopConfiguration)
fs.delete(new Path("/home/ariadne_storage"), true)


Intitializing Scala interpreter ...

Spark Web UI available at http://55a09705e3f4:4041
SparkContext available as 'sc' (version = 3.5.5, master = local[*], app id = local-1775571579571)
SparkSession available as 'spark'

import dev.cjfravel.ariadne.{Index, IndexCatalog}
import dev.cjfravel.ariadne.Index.DataFrameOps
import org.apache.spark.sql.types._
import org.apache.spark.sql.Row
spark: org.apache.spark.sql.SparkSession = org.apache.spark.sql.SparkSession@1e511eac
import org.apache.hadoop.fs.{FileSystem, Path}
fs: org.apache.hadoop.fs.FileSystem = org.apache.hadoop.hive.ql.io.ProxyLocalFileSystem@7f9c6b90
res0: Boolean = true

## Creating an Index

An index is defined by a name, source schema, file format, and optional read options.
Then you add files, choose index types, and call `update` to build.

In [3]:
val customerSchema = StructType(Seq(
  StructField("customer_id", IntegerType),
  StructField("name", StringType),
  StructField("email", StringType),
  StructField("region", StringType),
  StructField("signup_date", StringType)
))

// Create the index with schema, format, and CSV options
val customers = Index("api_customers", customerSchema, "csv", Map("header" -> "true"))

// Add source data files
customers.addFile("/home/data/customers_west.csv")
customers.addFile("/home/data/customers_east.csv")

// Add indexes on columns you'll query or join on
customers.addIndex("customer_id")
customers.addIndex("region")

// Build the index
customers.update
println("Index built successfully!")

Index built successfully!


customerSchema: org.apache.spark.sql.types.StructType = StructType(StructField(customer_id,IntegerType,true),StructField(name,StringType,true),StructField(email,StringType,true),StructField(region,StringType,true),StructField(signup_date,StringType,true))
customers: dev.cjfravel.ariadne.Index = Index(api_customers,Some(StructType(StructField(customer_id,IntegerType,true),StructField(name,StringType,true),StructField(email,StringType,true),StructField(region,StringType,true),StructField(signup_date,StringType,true))))

## Querying with locateFiles

`locateFiles` returns the set of source files containing matching values.
This is the core operation — it tells you *which files to read* without reading them.

In [4]:
// Single column lookup
val files1001 = customers.locateFiles(Map("customer_id" -> Array(1001)))
println(s"Customer 1001 is in: ${files1001.mkString(", ")}")

// Multi-value lookup
val filesMulti = customers.locateFiles(Map("customer_id" -> Array(1001, 2001)))
println(s"Customers 1001 or 2001 are in: ${filesMulti.mkString(", ")}")

// Multi-column lookup (AND semantics — files must match ALL columns)
val filesRegion = customers.locateFiles(Map(
  "customer_id" -> Array(1001, 1002, 2001),
  "region" -> Array("West")
))
println(s"West customers 1001/1002/2001: ${filesRegion.mkString(", ")}")

Customer 1001 is in: file:///home/data/customers_west.csv
Customers 1001 or 2001 are in: file:///home/data/customers_east.csv, file:///home/data/customers_west.csv
West customers 1001/1002/2001: file:///home/data/customers_west.csv


files1001: Set[String] = Set(file:///home/data/customers_west.csv)
filesMulti: Set[String] = Set(file:///home/data/customers_east.csv, file:///home/data/customers_west.csv)
filesRegion: Set[String] = Set(file:///home/data/customers_west.csv)

## Joining: Index.join(df)

The primary join API. Ariadne locates matching files, reads only those files,
then performs a Spark DataFrame join. Much faster than reading all files.

In [5]:
val orderSchema = StructType(Seq(
  StructField("order_id", IntegerType),
  StructField("customer_id", IntegerType),
  StructField("product_sku", StringType),
  StructField("amount", DoubleType),
  StructField("order_date", StringType)
))

val orders = Index("api_orders", orderSchema, "csv", Map("header" -> "true"))
orders.addFile("/home/data/orders_q1.csv")
orders.addFile("/home/data/orders_q2.csv")
orders.addIndex("customer_id")
orders.update

orderSchema: org.apache.spark.sql.types.StructType = StructType(StructField(order_id,IntegerType,true),StructField(customer_id,IntegerType,true),StructField(product_sku,StringType,true),StructField(amount,DoubleType,true),StructField(order_date,StringType,true))
orders: dev.cjfravel.ariadne.Index = Index(api_orders,Some(StructType(StructField(order_id,IntegerType,true),StructField(customer_id,IntegerType,true),StructField(product_sku,StringType,true),StructField(amount,DoubleType,true),StructField(order_date,StringType,true))))

In [6]:
// Join orders against customers — Ariadne reads only files with matching customer_ids
val ordersDf = spark.read.option("header", "true").schema(orderSchema)
  .csv("/home/data/orders_q1.csv", "/home/data/orders_q2.csv")

val enriched = customers.join(ordersDf, Seq("customer_id"), "inner")
enriched.select("customer_id", "name", "region", "order_id", "amount").show()

+-----------+--------------+------+--------+------+
|customer_id|          name|region|order_id|amount|
+-----------+--------------+------+--------+------+
|       1001|   Alice Smith|  West|   10001| 29.99|
|       1001|   Alice Smith|  West|   10008|  79.5|
|       1001|   Alice Smith|  West|   10006|199.99|
|       1002|   Bob Johnson|  West|   10003| 29.99|
|       1003|Carol Williams|  West|   10004|  79.5|
|       2001|    Dave Brown|  East|   10002|149.99|
|       2002|     Eve Davis|  East|   10005| 29.99|
|       2003|  Frank Miller|  East|   10007|149.99|
+-----------+--------------+------+--------+------+



ordersDf: org.apache.spark.sql.DataFrame = [order_id: int, customer_id: int ... 3 more fields]
enriched: org.apache.spark.sql.DataFrame = [customer_id: int, name: string ... 7 more fields]

## Joining: df.join(index)

The reverse direction — join a DataFrame against an index using the implicit conversion.
Import `Index.DataFrameOps` to enable `df.join(index, ...)`.

In [7]:
// Same join, but from the DataFrame side
val enriched2 = ordersDf.join(customers, Seq("customer_id"), "inner")
enriched2.select("customer_id", "name", "region", "order_id", "amount").show()

+-----------+--------------+------+--------+------+
|customer_id|          name|region|order_id|amount|
+-----------+--------------+------+--------+------+
|       1001|   Alice Smith|  West|   10001| 29.99|
|       1001|   Alice Smith|  West|   10008|  79.5|
|       1001|   Alice Smith|  West|   10006|199.99|
|       1002|   Bob Johnson|  West|   10003| 29.99|
|       1003|Carol Williams|  West|   10004|  79.5|
|       2001|    Dave Brown|  East|   10002|149.99|
|       2002|     Eve Davis|  East|   10005| 29.99|
|       2003|  Frank Miller|  East|   10007|149.99|
+-----------+--------------+------+--------+------+



enriched2: org.apache.spark.sql.DataFrame = [customer_id: int, order_id: int ... 7 more fields]

## Reconnecting to an Existing Index

Use `Index(name)` (without schema) to reconnect to a previously created index.
The schema and configuration are loaded from stored metadata.

In [8]:
// Reconnect to the customers index
val customers2 = Index("api_customers")
println(s"Name: ${customers2.name}")
println(s"Indexes: ${customers2.indexes}")
customers2.stats().show(false)


Name: api_customers
Indexes: Set(customer_id, region)
+---------+----------------------------------+-------------------+
|FileCount|customer_id                       |region             |
+---------+----------------------------------+-------------------+
|2        |{3, 4, 3.5, 3, 0.7071067811865476}|{1, 1, 1.0, 1, 0.0}|
+---------+----------------------------------+-------------------+



customers2: dev.cjfravel.ariadne.Index = Index(api_customers,None)

## Managing Indexes

Use `IndexCatalog` to list, check, and remove indexes.

In [9]:
// List all indexes
val allIndexes = IndexCatalog.list()
println(s"All indexes: ${allIndexes.mkString(", ")}")

// Check if an index exists
println(s"api_customers exists: ${IndexCatalog.exists("api_customers")}")
println(s"nonexistent exists: ${IndexCatalog.exists("nonexistent")}")

All indexes: api_customers, api_orders
api_customers exists: true
nonexistent exists: false


allIndexes: Seq[String] = WrappedArray(api_customers, api_orders)

## Index Maintenance: Compact and Vacuum

Delta Lake tables accumulate small files over time. Use `compact` to optimize
file sizes and `vacuum` to remove old versions.

In [10]:
// Compact the index Delta tables (runs Delta OPTIMIZE)
customers.compact()

// Vacuum old Delta versions (removes files older than 168 hours)
customers.vacuum(168)

Deleted 0 files and directories in a total of 1 directories.


## Deleting Files from an Index

When source files are removed or replaced, remove them from the index.

In [11]:
// Remove a file from the index (does not delete the source file)
// customers.deleteFiles(Seq("/home/data/customers_east.csv"))
// println("File removed from index")

// Uncomment the above to actually remove — left commented to keep the index intact

## Index Statistics

View per-column statistics about the index: min/max/avg array lengths,
which helps understand data distribution and index efficiency.

In [12]:
customers.stats().show()

+---------+--------------------+-------------------+
|FileCount|         customer_id|             region|
+---------+--------------------+-------------------+
|        2|{3, 4, 3.5, 3, 0....|{1, 1, 1.0, 1, 0.0}|
+---------+--------------------+-------------------+

